In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# Option si tu as category_encoders
from category_encoders import TargetEncoder

In [3]:
df = pd.read_csv('../data/data_propre_non_scale.csv')

# -----------------------------------------------
#                            TRAIN TEST SPLIT
# -----------------------------------------------

In [5]:
X = df.drop(columns=['target_is_fraud'])
y = df['target_is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# Liste des colonnes à supprimer car multicolinéarité forte
col_vif_trop_fort = [
    'terms_accepted_flag',
    'credit_score',
    'income_log',
    'avg_amount_30d_eur',
    'credit_score_norm',
    'income_estimate_alt_eur',
    'max_to_avg_ratio',
    'age',
    'tx_amount_total_30d_eur'
]

# Supprimer les colonnes dans X_train et X_test
X_train = X_train.drop(columns=col_vif_trop_fort)
X_test = X_test.drop(columns=col_vif_trop_fort)

In [ ]:
#  device_trust_z → 0
X_train['device_trust_z'] = X_train['device_trust_z'].fillna(0)
X_test['device_trust_z'] = X_test['device_trust_z'].fillna(0)


#  is_vpn_x_ip_risk → 0
X_train['is_vpn_x_ip_risk'] = X_train['is_vpn_x_ip_risk'].fillna(0)
X_test['is_vpn_x_ip_risk'] = X_test['is_vpn_x_ip_risk'].fillna(0)

In [ ]:
#Numériques (à scaler)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes catégorielles
onehot_cols = ["signup_source","os","browser","device_type","channel","plan_type","country"]
target_cols = ["payment_method","merchant_category","occupation","city"]

# Colonnes texte pour TF-IDF
text_cols = ["customer_note","last_ticket_subject"]

In [ ]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # médiane pour credit_score, device_trust_z=0 etc
    ("scaler", RobustScaler())                      # robuste aux outliers
])

onehot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("target", TargetEncoder())
])

text_transformers = [
    (
        f"tfidf_{col}",
        TfidfVectorizer(max_features=50),
        col
    )
    for col in text_cols
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("onehot", onehot_pipeline, onehot_cols),
        ("target", target_pipeline, target_cols),
        *text_transformers
    ],
    remainder="drop"
)

In [ ]:
X_train_enc = preprocessor.fit_transform(X_train, y_train)
X_test_enc = preprocessor.transform(X_test)

In [ ]:
clf2 = LogisticRegression(
    solver="saga",       # adapté pour penalty="l1"
    penalty="l1",        # Lasso, pour faire un peu de feature selection
    class_weight={0:1, 1:24},  # ajustable selon déséquilibre
    max_iter=500,
    random_state=42
)


In [ ]:
clf2.fit(X_train_enc, y_train)

In [ ]:
y_pred = clf2.predict(X_test_enc)
y_proba = clf2.predict_proba(X_test_enc)[:, 1]  # probabilité de fraude

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    auc
)
import matplotlib.pyplot as plt

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
recall = recall_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

y_proba = clf2.predict_proba(X_test_enc)[:,1]

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall_curve, precision_curve)


print("Recall :", recall)
print("Precision :", precision)
print("PR-AUC :", pr_auc)
print("ROC-AUC :", roc_auc)